### Import the required libraries

Import `requests` to call the CoinGecko API, `psycopg2` to write data to PostgreSQL, and `time` to pause between requests.

In [1]:
import requests
import psycopg2
import time

### Connect to PostgreSQL

Open a connection to the `streaming_db` database on `localhost:5432` using the configured PostgreSQL credentials. This connection is used to save cryptocurrency prices.

In [2]:
conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="streaming_db",
    user="postgres",
    password=""
)

### Prepare the database cursor and API endpoint

Create a cursor for executing SQL statements and set the CoinGecko simple-price endpoint used to request current prices.

In [3]:
cur = conn.cursor()
url = "https://api.coingecko.com/api/v3/simple/price"

### Continuously fetch and store cryptocurrency prices

Request Bitcoin and Ethereum prices in USD with a 10-second request timeout and print the response. If the response contains an `error` key, wait 30 seconds before retrying. Otherwise, skip entries without a USD price, insert the available prices into `crypto_price`, and commit the transaction. Pause for 15 seconds after each successful cycle. This loop runs until you interrupt the cell or an unhandled error occurs.

In [4]:
while True:
    r = requests.get(
        url,
        params={"ids": "bitcoin,ethereum", "vs_currencies": "usd"},
        timeout=10
    )

    data = r.json()
    print("Response:", data)

    if "error" in data:
        print("API error:", data)
        time.sleep(30)
        continue

    for coin, value in data.items():
        price = value.get("usd")

        if price is None:
            print(f"Skip {coin}, no USD price")
            continue

        cur.execute(
            "INSERT INTO crypto_price (coin, price_usd) VALUES (%s, %s)",
            (coin, price)
        )

    conn.commit()
    print("harga terbaru masuk DB")
    time.sleep(15)

Response: {'bitcoin': {'usd': 86075}, 'ethereum': {'usd': 2712.17}}
harga terbaru masuk DB
Response: {'bitcoin': {'usd': 86075}, 'ethereum': {'usd': 2712.17}}
harga terbaru masuk DB
Response: {'bitcoin': {'usd': 86075}, 'ethereum': {'usd': 2712.17}}
harga terbaru masuk DB
Response: {'bitcoin': {'usd': 86075}, 'ethereum': {'usd': 2712.17}}
harga terbaru masuk DB
Response: {'bitcoin': {'usd': 86015}, 'ethereum': {'usd': 2710.76}}
harga terbaru masuk DB
Response: {'bitcoin': {'usd': 86015}, 'ethereum': {'usd': 2710.76}}
harga terbaru masuk DB
Response: {'status': {'error_code': 429, 'error_message': "You've exceeded the Rate Limit. Please visit https://www.coingecko.com/en/api/pricing to subscribe to our API plans for higher rate limits."}}
Skip status, no USD price
harga terbaru masuk DB
Response: {'status': {'error_code': 429, 'error_message': "You've exceeded the Rate Limit. Please visit https://www.coingecko.com/en/api/pricing to subscribe to our API plans for higher rate limits."}}
S

KeyboardInterrupt: 